# Limpieza y Normalización de Datos
## Proyecto: Trayectorias Educativas en Argentina — Riesgo de Abandono Escolar por Establecimiento
### Fuente principal: Ministerio de Educación — Relevamiento Anual 2024 (Bases 2, 3 y 5)
### Fuentes secundarias: Matrícula agregada 2022–2024 · Tasas de abandono interanual 2003–2024

---

Este notebook documenta el proceso completo de auditoría, limpieza y normalización
de todos los datasets que alimentan el proyecto.

Cada sección detalla:
- **Qué** se transformó
- **Por qué** fue necesaria la transformación
- **Cuál fue el resultado** verificable

Las decisiones de limpieza relevantes quedan registradas también en `decisions_log.md`.

**Datasets procesados:**

| Dataset | Fuente | Granularidad |
|---|---|---|
| Matrícula total 2022, 2023, 2024 | Ministerio de Educación | Provincial |
| Matrícula por edad 2022, 2023, 2024 | Ministerio de Educación | Provincial |
| Base 2 — Matrícula RA 2024 | Relevamiento Anual 2024 | Por escuela |
| Base 3 — Trayectoria RA 2024 | Relevamiento Anual 2024 | Por escuela |
| Base 5 — Características RA 2024 | Relevamiento Anual 2024 | Por escuela |
| Abandono interanual 2012–2024 | DiNIEE / Red-FIE | Provincial |
| Abandono interanual 2003–2016 | DiNIEE / Red-FIE | Provincial |

**Output:** todos los datasets limpios se exportan a `data/clean/`.
A partir de este notebook, el resto del proyecto trabaja exclusivamente sobre esos archivos.
Los archivos en `data/raw/` nunca se modifican.

In [9]:
import pandas as pd
import os

# Rutas relativas — funcionan en cualquier máquina
RAW  = os.path.join("..", "data", "raw")
CLEAN = os.path.join("..", "data", "clean")

os.makedirs(CLEAN, exist_ok=True)

print("Setup OK")
print(f"  RAW  → {os.path.abspath(RAW)}")
print(f"  CLEAN → {os.path.abspath(CLEAN)}")

Setup OK
  RAW  → c:\Users\fedeo\Desktop\PROYECTO UNICEF\data\raw
  CLEAN → c:\Users\fedeo\Desktop\PROYECTO UNICEF\data\clean


## Sección 1 — Carga de datasets agregados

Se cargan los 6 datasets de matrícula provincial (2022–2024) y los 2 datasets
de tasas de abandono interanual (series 2012–2024 y 2003–2016).

**Consideraciones técnicas:**
- Los archivos CSV del Ministerio usan separador `;` y encoding `latin-1`.
  Ambos parámetros deben especificarse explícitamente o pandas los interpreta mal.
- Los archivos Excel de abandono tienen 10 filas de encabezado institucional
  antes de los datos reales. Se omiten con `skiprows=10`.
- Las rutas son relativas al notebook (`../data/raw/...`) para garantizar
  que el proyecto sea reproducible en cualquier máquina.

In [10]:
# --- Matrícula total ---
df_mat_2022 = pd.read_csv(os.path.join(RAW, "2022_relevamiento", "2022 Matricula - agregada(in).csv"), sep=";", encoding="latin-1")
df_mat_2023 = pd.read_csv(os.path.join(RAW, "2023_relevamiento", "2023 Matricula - agregada(in).csv"), sep=";", encoding="latin-1")
df_mat_2024 = pd.read_csv(os.path.join(RAW, "2024_relevamiento", "2024 Matricula - agregada(in).csv"), sep=";", encoding="latin-1")

# --- Matrícula por edad ---
df_edad_2022 = pd.read_csv(os.path.join(RAW, "2022_relevamiento", "2022 Matricula por edad - agregada(in).csv"), sep=";", encoding="latin-1")
df_edad_2023 = pd.read_csv(os.path.join(RAW, "2023_relevamiento", "2023 Matricula por edad - agregada(in).csv"), sep=";", encoding="latin-1")
df_edad_2024 = pd.read_csv(os.path.join(RAW, "2024_relevamiento", "2024 Matricula por edad - agregada(in).csv"), sep=";", encoding="latin-1")

# --- Abandono histórico ---
abandono_reciente  = pd.read_excel(os.path.join(RAW, "abandono_historico", "Tasa de Abandono Interanual 2024-2012 según división político-territorial.xlsx"), skiprows=10)
abandono_historico = pd.read_excel(os.path.join(RAW, "abandono_historico", "Tasa de Abandono Interanual 2016-2003 según estructura nacional homogenea.xlsx"), skiprows=10)

print("Datasets agregados cargados:")
print(f"  df_mat_2022:        {df_mat_2022.shape}")
print(f"  df_mat_2023:        {df_mat_2023.shape}")
print(f"  df_mat_2024:        {df_mat_2024.shape}")
print(f"  df_edad_2022:       {df_edad_2022.shape}")
print(f"  df_edad_2023:       {df_edad_2023.shape}")
print(f"  df_edad_2024:       {df_edad_2024.shape}")
print(f"  abandono_reciente:  {abandono_reciente.shape}")
print(f"  abandono_historico: {abandono_historico.shape}")

Datasets agregados cargados:
  df_mat_2022:        (1212, 99)
  df_mat_2023:        (1211, 103)
  df_mat_2024:        (1216, 104)
  df_edad_2022:       (20359, 43)
  df_edad_2023:       (20445, 43)
  df_edad_2024:       (21389, 43)
  abandono_reciente:  (38, 17)
  abandono_historico: (36, 17)


## Sección 2 — Limpieza de datasets agregados

Se realizan cuatro operaciones sobre los datasets cargados en la sección anterior:

1. **Normalización de nombres de columna** — se estandarizan a `snake_case` sin
   tildes ni caracteres especiales, para evitar errores al referenciarlas en código.
2. **Conversión de tipos** — las columnas numéricas de abandono se importan como
   `object` (texto) porque el Excel mezcla encabezados con valores. Se convierten
   a `float64` con `pd.to_numeric(errors='coerce')`: los valores no convertibles
   se transforman en `NaN`, haciéndolos visibles para la auditoría.
3. **Detección y eliminación de filas basura** — los Excel del Ministerio incluyen
   notas al pie (definición metodológica, fuente, fecha) que pandas importa como
   filas. Se identifican y eliminan mediante el criterio de que todas sus columnas
   numéricas sean `NaN`.
4. **Auditoría de nulos** — se verifica el conteo final de valores faltantes.
   Los nulos estructurales (provincias con estructura 6-6 no tienen `prim_7`;
   con estructura 7-5 no tienen `sec_7`) se conservan — son "no aplica", no datos faltantes.

In [11]:
# df_mat_2024 tiene dos columnas con mayúscula inicial — se corrigen
df_mat_2024 = df_mat_2024.rename(columns={
    "Departamento": "departamento",
    "Localizacion": "localizacion"
})

# Los datasets de matrícula por edad tienen tildes, ñ y espacios
# Esta función los estandariza a snake_case sin caracteres especiales
def limpiar_columnas(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.lower()
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("ascii")
        .str.replace(" ", "_", regex=False)
        .str.replace(r"[^a-z0-9_]", "", regex=True)
    )
    return df

df_edad_2022 = limpiar_columnas(df_edad_2022)
df_edad_2023 = limpiar_columnas(df_edad_2023)
df_edad_2024 = limpiar_columnas(df_edad_2024)

# Verificación
print("Primeras 4 columnas de cada dataset de matrícula:")
print(f"  df_mat_2022:  {list(df_mat_2022.columns[:4])}")
print(f"  df_mat_2023:  {list(df_mat_2023.columns[:4])}")
print(f"  df_mat_2024:  {list(df_mat_2024.columns[:4])}")
print(f"\nPrimeras 6 columnas — df_edad_2022:")
print(f"  {list(df_edad_2022.columns[:6])}")

Primeras 4 columnas de cada dataset de matrícula:
  df_mat_2022:  ['provincia', 'departamento', 'sector', 'ambito']
  df_mat_2023:  ['provincia', 'departamento', 'sector', 'ambito']
  df_mat_2024:  ['provincia', 'departamento', 'sector', 'ambito']

Primeras 6 columnas — df_edad_2022:
  ['provincia', 'departamento', 'sector', 'ambito', 'grado', '0anos']


In [12]:
# --- Abandono reciente (2012–2024) ---
abandono_reciente = abandono_reciente.rename(columns={
    "Unnamed: 0": "division",
    "Unnamed: 1": "estructura",
    "Total":      "prim_total",
    "1° Año":     "prim_1",
    "2° Año":     "prim_2",
    "3° Año":     "prim_3",
    "4° Año":     "prim_4",
    "5° Año":     "prim_5",
    "6° Año":     "prim_6",
    "7° Año":     "prim_7",
    "Total.1":    "sec_total",
    "7° Año.1":   "sec_7",
    "8° Año":     "sec_8",
    "9° Año":     "sec_9",
    "10° Año":    "sec_10",
    "11° Año":    "sec_11",
    "12° Año":    "sec_12",
})

# --- Abandono histórico (2003–2016) ---
abandono_historico = abandono_historico.rename(columns={
    "División":                                    "division",
    "Primaria (6 años)":                           "prim_total",
    "Unnamed: 2":                                  "prim_1",
    "Unnamed: 3":                                  "prim_2",
    "Unnamed: 4":                                  "prim_3",
    "Unnamed: 5":                                  "prim_4",
    "Unnamed: 6":                                  "prim_5",
    "Unnamed: 7":                                  "prim_6",
    "Secundaria                          (6 años)": "sec_total",
    "Secundaria - Ciclo Básico":                   "sec_cb_total",
    "Unnamed: 10":                                 "sec_7",
    "Unnamed: 11":                                 "sec_8",
    "Unnamed: 12":                                 "sec_9",
    "Secundaria - Ciclo Orientado":                "sec_co_total",
    "Unnamed: 14":                                 "sec_10",
    "Unnamed: 15":                                 "sec_11",
    "Unnamed: 16":                                 "sec_12",
})
abandono_historico = abandono_historico.drop(index=[0, 1]).reset_index(drop=True)

# --- Conversión a numérico ---
cols_reciente  = abandono_reciente.columns.drop(["division", "estructura"])
cols_historico = abandono_historico.columns.drop(["division"])

abandono_reciente[cols_reciente]   = abandono_reciente[cols_reciente].apply(pd.to_numeric, errors="coerce")
abandono_historico[cols_historico] = abandono_historico[cols_historico].apply(pd.to_numeric, errors="coerce")

print("Rename y conversión OK")
print(f"  abandono_reciente columnas:  {list(abandono_reciente.columns)}")
print(f"  abandono_historico columnas: {list(abandono_historico.columns)}")

Rename y conversión OK
  abandono_reciente columnas:  ['division', 'estructura', 'prim_total', 'prim_1', 'prim_2', 'prim_3', 'prim_4', 'prim_5', 'prim_6', 'prim_7', 'sec_total', 'sec_7', 'sec_8', 'sec_9', 'sec_10', 'sec_11', 'sec_12']
  abandono_historico columnas: ['division', 'prim_total', 'prim_1', 'prim_2', 'prim_3', 'prim_4', 'prim_5', 'prim_6', 'sec_total', 'sec_cb_total', 'sec_7', 'sec_8', 'sec_9', 'sec_co_total', 'sec_10', 'sec_11', 'sec_12']


In [13]:
# Filas donde TODAS las columnas numéricas son NaN son pie de página del Excel
cols_num_rec  = abandono_reciente.columns.drop(["division", "estructura"])
cols_num_hist = abandono_historico.columns.drop(["division"])

abandono_reciente  = abandono_reciente.dropna(subset=cols_num_rec, how="all").reset_index(drop=True)
abandono_historico = abandono_historico.dropna(subset=cols_num_hist, how="all").reset_index(drop=True)

print("Shapes post-limpieza (esperado: 27 × 17 en ambos):")
print(f"  abandono_reciente:  {abandono_reciente.shape}")
print(f"  abandono_historico: {abandono_historico.shape}")

Shapes post-limpieza (esperado: 27 × 17 en ambos):
  abandono_reciente:  (27, 17)
  abandono_historico: (27, 17)


## Sección 3 — Carga de bases por escuela (Relevamiento Anual 2024)

Se cargan las tres bases del Relevamiento Anual 2024 que operan a nivel de establecimiento.
Son la fuente principal del modelo de Machine Learning.

| Base | Archivo | Contenido |
|---|---|---|
| Base 2 | `2024 Matricula - agregada(in).csv` | Matrícula total y por sexo, repetidores, sobreedad, secciones |
| Base 3 | `2024 Trayectoria - agregada(in).csv` | Flujo de alumnos: entrados, salidos con pase, salidos sin pase (abandono), promovidos, repitentes |
| Base 5 | `2024 Caracteristicas - agregada(in).csv` | Servicios, equipamiento, conectividad y características organizativas del establecimiento |

**Base 3 es el corazón del modelo.** Contiene las variables `ssp_N` (salidos sin pase
por grado), que son la materia prima para construir la variable target `tasa_abandono`.

Los tres archivos usan separador `;` y encoding `latin-1`, igual que el resto
de los datasets del Ministerio.

In [14]:
RAW_2024 = os.path.join(RAW, "2024_relevamiento")

base2 = pd.read_csv(os.path.join(RAW_2024, "2024 Matricula - agregada(in).csv"),       sep=";", encoding="latin-1")
base3 = pd.read_csv(os.path.join(RAW_2024, "2024 Trayectoria - agregada(in).csv"),     sep=";", encoding="latin-1")
base5 = pd.read_csv(os.path.join(RAW_2024, "2024 Caracteristicas - agregada(in).csv"), sep=";", encoding="latin-1")

print("Bases por escuela cargadas:")
print(f"  Base 2 (matrícula):       {base2.shape}")
print(f"  Base 3 (trayectoria):     {base3.shape}")
print(f"  Base 5 (características): {base5.shape}")

Bases por escuela cargadas:
  Base 2 (matrícula):       (1216, 104)
  Base 3 (trayectoria):     (1209, 288)
  Base 5 (características): (1216, 66)


## Sección 4 — Limpieza de bases por escuela

Se realizan tres operaciones sobre las bases del Relevamiento Anual 2024:

1. **Normalización de nombres de columna** — se aplica la misma función `limpiar_columnas()`
   usada en la sección anterior para estandarizar a `snake_case` sin tildes ni caracteres especiales.

2. **Auditoría de nulos** — se revisa el porcentaje de valores faltantes por columna.
   En bases con 288 columnas es esperable encontrar columnas con alta proporción de NaN
   (grados que no aplican a ciertos establecimientos, por ejemplo). Se documentan
   los patrones encontrados.

3. **Identificación de columna clave** — se verifica la presencia del identificador
   único de establecimiento (`cueanexo` o equivalente) que permitirá unir las tres
   bases en la etapa de feature engineering.

**Nota sobre Base 3:** tiene 1.209 filas vs. 1.216 de Base 2 y Base 5.
Las 7 escuelas faltantes no reportaron datos de trayectoria en el RA 2024.
Este diferencial se gestiona en la etapa de unión de bases (notebook 04).

In [15]:
base2 = limpiar_columnas(base2)
base3 = limpiar_columnas(base3)
base5 = limpiar_columnas(base5)

print("Primeras 6 columnas de cada base:")
print(f"  Base 2: {list(base2.columns[:6])}")
print(f"  Base 3: {list(base3.columns[:6])}")
print(f"  Base 5: {list(base5.columns[:6])}")

Primeras 6 columnas de cada base:
  Base 2: ['provincia', 'departamento', 'sector', 'ambito', 'localizacion', 'lactantes']
  Base 3: ['provincia', 'departamento', 'sector', 'ambito', 'inicial_1', 'inicial_2']
  Base 5: ['provincia', 'departamento', 'sector', 'ambito', 'localizacion', 'electricidadsi']


In [16]:
# Con 288 columnas no podemos revisar una por una
# Mostramos solo las columnas con más del 5% de nulos
umbral = 0.05

for nombre, df in [("Base 2", base2), ("Base 3", base3), ("Base 5", base5)]:
    nulos_pct = df.isnull().mean()
    problemas = nulos_pct[nulos_pct > umbral].sort_values(ascending=False)
    print(f"\n{nombre} — columnas con más del 5% de nulos: {len(problemas)}")
    if len(problemas) > 0:
        print(problemas.to_string())


Base 2 — columnas con más del 5% de nulos: 0

Base 3 — columnas con más del 5% de nulos: 0

Base 5 — columnas con más del 5% de nulos: 0


In [17]:
# El identificador único de establecimiento es lo que va a permitir
# unir Base 2, 3 y 5 en una sola tabla para el modelo
# Buscamos variantes del nombre porque el Ministerio no siempre es consistente

posibles_ids = ["cueanexo", "cue_anexo", "cue", "codigo_establecimiento", "id_establecimiento"]

for nombre, df in [("Base 2", base2), ("Base 3", base3), ("Base 5", base5)]:
    encontrados = [col for col in df.columns if col in posibles_ids]
    print(f"  {nombre}: {encontrados if encontrados else '⚠ ninguno encontrado'}")

  Base 2: ⚠ ninguno encontrado
  Base 3: ⚠ ninguno encontrado
  Base 5: ⚠ ninguno encontrado


In [20]:
print(base2.columns.tolist())

['provincia', 'departamento', 'sector', 'ambito', 'localizacion', 'lactantes', 'deambulantes', 's2', 's3', 's4', 's5', '_1', '_2', '_3', '_4', '_5', '_6', '_7', '_8', '_9', '_10', '_11', '_12', '_1314', '_20', '_snu', 'v_lactantes', 'v_deambulantes', 'v_s2', 'v_s3', 'v_s4', 'v_s5', 'v_1', 'v_2', 'v_3', 'v_4', 'v_5', 'v_6', 'v_7', 'v_8', 'v_9', 'v_10', 'v_11', 'v_12', 'v_1314', 'v_20', 'v_snu', 'r_1', 'r_2', 'r_3', 'r_4', 'r_5', 'r_6', 'r_7', 'r_8', 'r_9', 'r_10', 'r_11', 'r_12', 'r_1314', 'r_20', 's_lactantes', 's_deambulantes', 's_s2', 's_s3', 's_s4', 's_s5', 's_1', 's_2', 's_3', 's_4', 's_5', 's_6', 's_7', 's_8', 's_9', 's_10', 's_11', 's_12', 's_1314', '_seclactantes', '_secdeambu', '_secs2', '_secs3', '_secs4', '_secs5', '_sec1', '_sec2', '_sec3', '_sec4', '_sec5', '_sec6', '_sec7', '_sec8', '_sec9', '_sec10', '_sec11', '_sec12', '_sec1314', '_sec20', 'multi_ini', 'multi_pri', 'multi_sec', 'multinivel']
